<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='24.%20advanced_patterns.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Previous</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <span style='padding:6px 14px; background:#e9ecef; color:#6c757d; border-radius:4px; text-decoration:none; font-weight:bold;'>Next →</span>
</div>


# Chapter 25: Access Control Using Casbin

Authentication answers **"Who are you?"** Access control answers **"What are you allowed to do?"** This chapter introduces access control from first principles, then shows how Casbin can help you enforce policies in a FastAPI app using dependencies.


## The Big Picture

Beginner APIs often stop after login, but real systems need a second decision after identity is known.

Examples:

- an authenticated customer can read their own orders but not another customer's orders,
- an editor can update articles, but a viewer cannot,
- an admin can manage users and policies.

This is the job of **authorization** or **access control**. Casbin is a policy engine that helps you separate these permission rules from route code so they stay easier to understand and change.


## Core Concepts

The easiest way to read this section is to keep one plain-language question in view: what is Core Concepts actually responsible for? Once that job is clear, the terminology stops feeling arbitrary and the details start to line up.

- **Authentication**: proving who the user is.
- **Authorization / access control**: deciding what that user can do.
- **Policy**: a rule such as "admins can delete reports."
- **Enforcer**: the Casbin object that loads rules and answers permission questions.
- **Model**: the file that defines how requests, policies, and matching should be interpreted.
- **RBAC (Role-Based Access Control)**: permissions are grouped by role, such as `admin` or `editor`.
- **ABAC (Attribute-Based Access Control)**: permissions depend on attributes such as owner, department, or resource status.
- **Dependency-based enforcement**: checking permission inside a FastAPI dependency before the endpoint continues.

A beginner-friendly mental model is: **Casbin is a rulebook plus a referee**.


## Syntax & First Usage

The problem Casbin solves is not "how do I write `if user.role == 'admin'`?" You can already do that. The problem is: **what happens when permission rules grow and spread across many files?**

Casbin gives you a central place for authorization rules. A tiny RBAC setup often uses:

- a `model.conf` file to describe the rule logic,
- a `policy.csv` file to store permissions,
- an `Enforcer` to check requests.


In [ ]:
import casbin

enforcer = casbin.Enforcer('model.conf', 'policy.csv')

allowed = enforcer.enforce('admin', 'reports', 'delete')
denied = enforcer.enforce('viewer', 'reports', 'delete')

print('admin can delete reports:', allowed)
print('viewer can delete reports:', denied)


## Deep Dive with Code Examples

Here is a very small model and policy. Read it slowly. The request asks a question shaped like `(subject, object, action)`.

**`model.conf`**
```ini
[request_definition]
r = sub, obj, act

[policy_definition]
p = sub, obj, act

[role_definition]
g = _, _

[policy_effect]
e = some(where (p.eft == allow))

[matchers]
m = g(r.sub, p.sub) && r.obj == p.obj && r.act == p.act
```

**`policy.csv`**
```csv
p, admin, reports, read
p, admin, reports, delete
p, editor, reports, read
g, alice, admin
g, bob, editor
```

In plain English:

- `p` lines define permissions for roles,
- `g` lines assign users to roles,
- the matcher says a request is allowed when the user's role matches and the object/action match too.


In [ ]:
from fastapi import Depends, FastAPI, HTTPException

app = FastAPI(title='Casbin Demo')

class User:
    def __init__(self, username: str, role: str):
        self.username = username
        self.role = role

def get_current_user() -> User:
    return User(username='alice', role='admin')

def authorize(resource: str, action: str):
    def dependency(user: User = Depends(get_current_user)):
        if not enforcer.enforce(user.role, resource, action):
            raise HTTPException(status_code=403, detail='You do not have permission to perform this action.')
        return user
    return dependency

@app.delete('/reports/{report_id}')
def delete_report(report_id: int, user: User = Depends(authorize('reports', 'delete'))):
    return {'message': f'Report {report_id} deleted by {user.username}'}


## RBAC vs ABAC

The hard part here is usually not the syntax but the boundary between similar ideas. Keep comparing the job each option does best, the tradeoff it introduces, and the clues that tell you which one fits the situation in front of you.

| Model | Main idea | Example |
| --- | --- | --- |
| RBAC | Permissions are grouped by role | `admin` can delete reports |
| ABAC | Permissions depend on attributes | a user can edit a document only if `user.id == document.owner_id` |

RBAC is usually easier for beginners because it is simple and predictable. ABAC becomes helpful when the rule depends on details of the user or resource, not just a role name.


## Deep Dive with Code Examples

Imagine an app where both Alice and Bob are `editors`, but each editor may only change documents they own. A plain role check is no longer enough.

That is where ABAC becomes useful. Instead of asking only:

- "Is this user an editor?"

you ask:

- "Is this user an editor **and** the owner of this document?"

Casbin can express these richer rules, but it is wise to start with RBAC first, understand it well, and only then add attribute-based rules.


## The "What If?" Experimentation Section

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

1. Add a `viewer` role that may only read reports. What lines would you add to `policy.csv`?
2. Change the dependency so the denied response uses your custom error shape from Chapter 22.
3. Imagine a user is both `editor` and `auditor`. How should your policy model handle multiple roles?
4. What extra resource data would you need for an ABAC rule like "users can edit only their own documents"?
5. If policies change often, why is a central rule engine easier to maintain than scattered `if` statements?


## Real-World Project Link

Start with the official Python Casbin project:

- https://github.com/casbin/pycasbin

Its examples are useful for seeing how models, policies, and enforcement fit together before you adapt the ideas to a FastAPI codebase.


## Chapter Summary & Cheat Sheet

### Summary
Access control decides what an authenticated user may do. Casbin helps by keeping permission rules in policies instead of burying them inside route logic. In FastAPI, dependency-based enforcement is a clean way to stop unauthorized requests before endpoint code runs.

### Cheat sheet

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

- **Authentication** = who the user is.
- **Authorization** = what the user may do.
- **Casbin Enforcer** answers permission checks.
- A common request shape is **`(subject, object, action)`**.
- **RBAC** groups permissions by role.
- **ABAC** uses attributes such as owner, team, or status.
- FastAPI dependencies are a great place to enforce access rules consistently.


## Practice Prompts

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

1. Explain authentication vs authorization in one sentence each.
2. What problem does Casbin solve beyond simple `if role == 'admin'` checks?
3. Why is dependency-based enforcement a good fit for FastAPI?
4. Write a sample policy rule for a `manager` who can approve invoices.
5. Describe one situation where ABAC is a better fit than RBAC.


<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='24. advanced_patterns.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <span style='color:gray; font-size:1.05em;'>Next</span>
</div>
